# Lab 04 — Window functions (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: usar `OVER (PARTITION BY ... ORDER BY ...)` para ranking e acumulado sem colapsar linhas.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
import pandas as pd
pedidos = pd.DataFrame([
    (1,'SP','eletronicos',1200.0,1),(2,'SP','livros',50.0,2),(3,'RJ','livros',30.0,1),
    (4,'MG','casa',80.0,3),(5,'SP','eletronicos',800.0,2),(6,'RJ','casa',150.0,4),
    (7,'SP','livros',45.0,1),(8,'MG','eletronicos',600.0,3),(9,'RJ','eletronicos',900.0,2),
    (10,'SP','casa',200.0,5),(11,'MG','livros',25.0,4),(12,'SP','eletronicos',1500.0,1),
    (13,'RJ','livros',60.0,5),(14,'MG','casa',120.0,3),(15,'SP','livros',40.0,2),
], columns=['id','estado','categoria','valor','cliente_id'])
len(pedidos)

## 1. Total do estado ao lado de cada linha (sem colapsar)

In [ ]:
duckdb.query('''
    SELECT id, estado, valor,
           SUM(valor) OVER (PARTITION BY estado) AS total_estado
    FROM pedidos
    ORDER BY estado, valor DESC
''').to_df().head(8)

## 2. Ranking por estado (ROW_NUMBER)

In [ ]:
duckdb.query('''
    SELECT id, estado, valor,
           ROW_NUMBER() OVER (PARTITION BY estado ORDER BY valor DESC) AS posicao
    FROM pedidos
    ORDER BY estado, posicao
''').to_df().head(8)

## 3. Total acumulado por id (running total)

In [ ]:
duckdb.query('''
    SELECT id, valor, SUM(valor) OVER (ORDER BY id) AS acumulado
    FROM pedidos ORDER BY id
''').to_df().head(6)

## 4. Sua vez (mini-desafio)
Traga o **pedido de maior valor de cada estado** (colunas `estado`, `id`, `valor`), ordenado por `valor` desc. (Dica: `ROW_NUMBER` numa CTE + filtro `= 1`.) Verifique.

In [ ]:
resposta = duckdb.query('''
    WITH r AS (
        SELECT estado, id, valor,
               ROW_NUMBER() OVER (PARTITION BY estado ORDER BY valor DESC) AS rn
        FROM pedidos
    )
    SELECT estado, id, valor FROM r WHERE rn = 1 ORDER BY valor DESC
''').to_df()
resposta

In [ ]:
def verificar(df):
    try:
        assert list(df['id']) == [12, 9, 8], 'Top por estado: SP=12, RJ=9, MG=8.'
        print('\u2705 Correto! ROW_NUMBER + filtro rn=1.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)